# Eksplorasi PySpark — NovelLens

Notebook untuk memeriksa data lake secara interaktif. Berjalan di container `notebook`,
yang menjadi **driver Spark tersendiri** — terpisah dari container `pipeline` yang dipakai
job batch, sehingga eksperimen di sini tidak mengganggu jalannya pipeline.

Buka lewat <http://localhost:8888>.

| Zona | Isi |
|---|---|
| `s3a://raw-data/` | CSV mentah apa adanya (Fase 1) |
| `s3a://clean-data/novels_clean` | Parquet bersih siap analisis (Fase 2) |
| `s3a://curated-data/analytics/` | Hasil kueri Spark SQL (Fase 3) |
| `s3a://curated-data/models/` | Cadangan artefak model (Fase 4) |

## 1. Menyalakan SparkSession

Selalu lewat `build_spark_session()`, jangan `SparkSession.builder` langsung — factory itu
yang memasang konfigurasi konektor S3A ke MinIO dan `spark.driver.host` yang benar.
Tanpa keduanya, job menggantung di stage pertama atau gagal dengan `403 Forbidden`.

In [1]:
from src.common.config import get_settings
from src.common.spark_session import build_spark_session

settings = get_settings()
spark = build_spark_session("notebook")

print(f"Spark   : {spark.version}")
print(f"Master  : {settings.spark.master_url}")
print(f"MinIO   : {settings.minio.endpoint}")
print("Spark UI: http://localhost:8080")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/30 05:44:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-08-30 05:45:00 | INFO     | src.common.spark_session | SparkSession aktif | app=notebook | master=spark://spark-master:7077 | versi=3.5.3
Spark   : 3.5.3
Master  : spark://spark-master:7077
MinIO   : http://minio:9000
Spark UI: http://localhost:8080


## 2. Isi data lake

In [2]:
from src.common.storage import list_objects

for bucket in settings.minio.buckets:
    objects = list_objects(bucket)
    total_mb = sum(o["Size"] for o in objects) / 1_048_576
    print(f"{bucket:<14} {len(objects):>4} objek  {total_mb:>8.2f} MB")

raw-data          4 objek      6.98 MB
clean-data        5 objek     11.50 MB
curated-data     49 objek     64.07 MB


## 3. Membaca data bersih

`novels_clean` adalah keluaran Fase 2: sentinel sudah dinormalkan jadi NULL, duplikat dibuang
berdasarkan `series_id`, sinopsis sudah tergabung, dan korpus teks sudah ditokenisasi.

In [3]:
clean_uri = settings.minio.s3a_uri(settings.minio.bucket_clean, "novels_clean")
novels = spark.read.parquet(clean_uri).cache()
novels.createOrReplaceTempView("novels")

print(f"{novels.count():,} baris x {len(novels.columns)} kolom")
novels.select(
    "title", "primary_genre", "chapter_count", "rating_score",
    "weighted_rating", "synopsis_word_count", "token_count",
).show(8, truncate=32)

26/08/30 05:45:09 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/08/30 05:45:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


4,461 baris x 47 kolom
+--------------------------------+-------------+-------------+------------+---------------+-------------------+-----------+
|                           title|primary_genre|chapter_count|rating_score|weighted_rating|synopsis_word_count|token_count|
+--------------------------------+-------------+-------------+------------+---------------+-------------------+-----------+
|The Ex-Inquisitor’s Exorcism ...|       Action|          200|         3.7|         3.6346|                 24|         48|
|Bound to the Spoiled Concubin...|       Comedy|          273|         2.2|         3.0026|                192|        105|
|Tang Yin’s Adventure in Anoth...|       Action|         2035|         3.4|         3.4301|                 83|         85|
|No One Would Ever Suspect tha...|        Adult|         NULL|         3.0|         3.3228|                 48|         52|
|            The Underground City|      Mystery|         NULL|         4.4|         3.8613|                16

In [4]:
novels.printSchema()

root
 |-- url: string (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- series_id: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- type_origin: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- language: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- original_publisher: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- status_coo: string (nullable = true)
 |-- chapter_count: integer (nullable = true)
 |-- is_completed: integer (nullable = true)
 |-- licensed: integer (nullable = true)
 |-- completely_translated: integer (nullable = true)
 |-- release_frequency: string (nullable = true)
 |-- rating_score: double (nullable = true)
 |-- rating_votes: integer (nullable = true)
 |-- activity_week_rank: integer (nullable = true)
 |-- activity_month_rank: integer (nullable = true)
 |-- activity_all_rank: integer (nullable = true)
 |-- reading_list_count: integer (nullable 

## 4. Kueri Spark SQL bebas

Ubah kueri di bawah untuk eksplorasi sendiri. Ingat dua aturan analitik dari
`Doc/data_contract.md`:

1. Pakai `weighted_rating`, bukan `rating_score` mentah, untuk peringkat apa pun.
2. Batasi populasi ke `is_rated_reliable` (>= 10 suara) saat menghitung rata-rata rating.

In [5]:
spark.sql("""
    SELECT genre,
           COUNT(*)                        AS n_novels,
           ROUND(AVG(weighted_rating), 4)  AS avg_rating,
           ROUND(AVG(chapter_count), 1)    AS avg_chapters
    FROM (SELECT explode(genre_list) AS genre, weighted_rating, chapter_count
          FROM novels WHERE is_rated_reliable)
    GROUP BY genre
    HAVING COUNT(*) >= 25
    ORDER BY avg_rating DESC
    LIMIT 15
""").show(truncate=False)

+-------------+--------+----------+------------+
|genre        |n_novels|avg_rating|avg_chapters|
+-------------+--------+----------+------------+
|Horror       |75      |3.7179    |412.7       |
|Mystery      |212     |3.68      |504.0       |
|Historical   |129     |3.6581    |354.0       |
|Supernatural |356     |3.6301    |502.1       |
|Tragedy      |154     |3.6056    |373.1       |
|Psychological|178     |3.5996    |376.8       |
|Slice of Life|482     |3.5952    |322.2       |
|Sci-fi       |141     |3.5912    |643.3       |
|Wuxia        |37      |3.5779    |583.5       |
|Comedy       |778     |3.5712    |420.4       |
|Xianxia      |145     |3.5688    |880.2       |
|Drama        |669     |3.5533    |324.5       |
|Adventure    |753     |3.5278    |634.5       |
|Fantasy      |1245    |3.5171    |487.8       |
|Romance      |1140    |3.5142    |348.8       |
+-------------+--------+----------+------------+



In [13]:
spark.sql(
    """
    select * from novels
    """
).show(truncate=False)

Py4JJavaError: An error occurred while calling o107.showString.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.comma

## 5. Memeriksa rencana eksekusi

Berguna untuk memastikan Spark benar-benar mem-*pushdown* filter ke Parquet
dan tidak melakukan shuffle yang tidak perlu.

In [6]:
(
    novels.filter("is_rated_reliable AND chapter_count > 500")
    .groupBy("type_origin")
    .avg("weighted_rating")
    .explain("formatted")
)

== Physical Plan ==
AdaptiveSparkPlan (10)
+- HashAggregate (9)
   +- Exchange (8)
      +- HashAggregate (7)
         +- Project (6)
            +- Filter (5)
               +- InMemoryTableScan (1)
                     +- InMemoryRelation (2)
                           +- * ColumnarToRow (4)
                              +- Scan parquet  (3)


(1) InMemoryTableScan
Output [4]: [chapter_count#12, is_rated_reliable#42, type_origin#5, weighted_rating#41]
Arguments: [chapter_count#12, is_rated_reliable#42, type_origin#5, weighted_rating#41], [isnotnull(is_rated_reliable#42), isnotnull(chapter_count#12), is_rated_reliable#42, (chapter_count#12 > 500)]

(2) InMemoryRelation
Arguments: [url#0, title#1, genres#2, series_id#3, type#4, type_origin#5, tags#6, language#7, authors#8, original_publisher#9, year#10, status_coo#11, chapter_count#12, is_completed#13, licensed#14, completely_translated#15, release_frequency#16, rating_score#17, rating_votes#18, activity_week_rank#19, activity_month_ra

## 6. Membaca hasil analitik Fase 3

Tabel-tabel ini yang dibaca dashboard. Daftar lengkapnya diambil dari isi bucket.

In [7]:
prefixes = sorted({
    o["Key"].split("/")[1]
    for o in list_objects(settings.minio.bucket_curated, "analytics/")
})
print(f"{len(prefixes)} tabel analitik:")
print("  " + "\n  ".join(prefixes))

20 tabel analitik:
  best_rated_tags
  correlation_metrics
  decade_trend
  genre_combinations
  genre_frequency
  genre_origin_matrix
  genre_pairs
  genre_ranking
  length_vs_rating
  origin_type_summary
  quality_report
  stat_descriptive
  stat_outlier_novels
  stat_outlier_summary
  synopsis_discriminative_tokens
  text_correlation
  text_richness_vs_rating
  top_tags
  top_tokens
  viz_novels


In [8]:
table = "genre_ranking"  # ganti dengan nama tabel mana pun di atas
uri = settings.minio.s3a_uri(settings.minio.bucket_curated, f"analytics/{table}")
spark.read.parquet(uri).show(12, truncate=False)

+-------------+--------+-------------------+------------+---------------+--------------+--------------+
|genre        |n_novels|avg_weighted_rating|avg_chapters|median_chapters|rank_by_rating|rank_by_length|
+-------------+--------+-------------------+------------+---------------+--------------+--------------+
|Horror       |75      |3.7179             |412.7       |239            |1             |15            |
|Mystery      |212     |3.68               |504.0       |247            |2             |10            |
|Historical   |129     |3.6581             |354.0       |222            |3             |19            |
|Supernatural |356     |3.6301             |502.1       |254            |4             |11            |
|Tragedy      |154     |3.6056             |373.1       |150            |5             |17            |
|Psychological|178     |3.5996             |376.8       |134            |6             |16            |
|Slice of Life|482     |3.5952             |322.2       |149    

## 7. Mencoba model terlatih

Inferensi tidak memakai Spark sama sekali — artefak dimuat langsung dari `Model/artifacts/`
agar responsnya milidetik.

In [9]:
from src.common.model_registry import artifacts_available, load_registry

sinopsis = (
    "Setelah dikhianati sekutunya dan tewas di puncak menara, seorang pendekar "
    "terbangun kembali sepuluh tahun lebih muda dan berniat menguasai teknik "
    "pedang terlarang sebelum musuh lamanya bangkit."
)
tags = "Time Travel, Weak to Strong, Cultivation, Male Protagonist, Revenge"

if not artifacts_available():
    print("Artefak belum ada. Latih dulu: python Model/train.py")
else:
    registry = load_registry()
    for name in registry.genre_model_names:
        info = registry.genre_model_info(name)
        hasil = registry.predict_genres(sinopsis, tags=tags, model=name, top_k=8)
        terpilih = hasil.loc[hasil["diprediksi"], "genre"].tolist()
        print(f"{info['label']:<22} macro-F1 {info['f1_macro']:.4f} -> {', '.join(terpilih)}")

2026-08-30 05:45:51 | INFO     | model_registry         | Registry dimuat: 4,460 novel, embedding (4460, 200), 24 genre, 3 model genre.
Linear SVM             macro-F1 0.5377 -> Action, Fantasy, Martial Arts, Adventure, Xianxia
Logistic Regression    macro-F1 0.5321 -> Action, Fantasy, Martial Arts, Adventure
BiLSTM + Attention     macro-F1 0.4812 -> Action, Xuanhuan, Martial Arts, Fantasy, Xianxia, Harem, Adventure


In [10]:
registry.recommend_by_text(tags, k=5)[
    ["title", "similarity", "primary_genre", "rating_score", "chapter_count"]
]

,title,similarity,primary_genre,rating_score,chapter_count
0,American Justice,0.5527,Drama,2.0,358.0
1,I have an External Incarnation,0.4951,Action,3.2,2086.0
2,I Became the Archmage That Transcendents Are O...,0.4828,Adventure,3.2,2.0
3,Api di Bukit Menoreh,0.4712,Action,3.0,396.0
4,Omnipotent Overlord,0.4644,Action,3.1,1558.0


## 8. Menutup session

Jalankan sebelum menutup notebook agar slot executor di cluster dilepas —
kalau tidak, job Fase 1–4 berikutnya bisa antre menunggu sumber daya.

In [11]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
